# TransitERA: Unsupervised Station Typology Discovery & Spatial Econometric Land Value Regression
**Kategori:** MAPID WebGIS Competition 2026 — Surabaya Raya Mass Transit Corridor  
**Fokus Penelitian:** Audit Spasial Multikriteria 15 Simpul Stasiun, Klastering Tanpa Pengawasan (*Unsupervised Clustering*), dan Pemodelan Kenaikan Nilai Tanah (%ΔNJOP).

---
### Ringkasan Pendekatan & Metodologi Ilmiah
Notebook ini dirancang untuk memastikan bahwa **kategori/tipologi simpul stasiun tidak ditentukan secara subjektif di awal (*no pre-destined categories*)**, melainkan ditemukan secara organik dari karakteristik empiris data spasial:
1. **Multi-Source Spatial Feature Ingestion**: Menggabungkan data spasial GEO MAPID (titik halte, poligon demografi kelurahan, kerentanan banjir, pusat perbelanjaan/mall, SES, dan harga tanah riil).
2. **5D TOD Spatial Feature Engineering**: Menghitung metrik *Density*, *Diversity*, *Design*, *Destination Accessibility*, dan *Distance to Transit* untuk seluruh 15 stasiun rel Surabaya.
3. **Dimensionality Reduction (PCA)**: Mengidentifikasi sumbu varians utama yang menggerakkan karakteristik stasiun di Surabaya.
4. **Unsupervised Clustering (K-Means & Hierarchical)**: Menemukan $k$ klaster optimal menggunakan Silhouette Analysis & Elbow Method, serta menginterpretasi profil centroid untuk merumuskan tipologi alami.
5. **Spatial Econometric Regression (SDM & OLS)**: Memodelkan elastisitas apresiasi nilai tanah (%ΔNJOP) terhadap kesiapan TOD dengan memperhitungkan efek limpahan spasial (*spatial spillover*).
6. **Ekspor Konfigurasi Terkalibrasi**: Menyimpan hasil audit ke `webdev/backend/app/data/calibrated_models.json` untuk dikonsumsi langsung oleh sistem WebGIS.


In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import shape, Point
import openpyxl
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Setup styling grafik
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 6)

DATA_DIR = os.path.abspath(os.path.join("..", "webdev", "backend", "app", "data", "spatial"))
print(f"Direktori Data Spasial: {DATA_DIR}")
assert os.path.exists(DATA_DIR), "Direktori data spasial tidak ditemukan!"


## Bab 1: Pemuatan Dataset Spasial Empiris Kota Surabaya
Kami memuat 9 dataset spasial riil yang mencakup batas stasiun, sebaran halte transit, zonasi demografi penduduk, kerentanan banjir, persebaran pusat perbelanjaan, SES, dan proksi NJOP per kecamatan.


In [ ]:
def load_geojson(filename):
    filepath = os.path.join(DATA_DIR, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

stations_raw = load_geojson("stasiun_surabaya.geojson")
demo_raw = load_geojson("demografi_surabaya.geojson")
halte_raw = load_geojson("halte_surabaya.geojson")
banjir_raw = load_geojson("banjir_surabaya.geojson")
malls_raw = load_geojson("PUSAT PERBELANJAAN DI KOTA SURABAYA TAHUN 2025.geojson")
ses_raw = load_geojson("STATUS EKONOMI DAN SOSIAL - SOCIOECONOMIC STATUS (SES) KOTA SURABAYA TAHUN 2024.geojson")
properti_raw = load_geojson("HARGA PROPERTI DI KOTA SURABAYA TAHUN 2024.geojson")

print(f"Total layer stasiun dimuat: {len(stations_raw['features'])} entri")
print(f"Total poligon demografi: {len(demo_raw['features'])} kelurahan")
print(f"Total titik halte transit: {len(halte_raw['features'])} halte")
print(f"Total poligon risiko banjir: {len(banjir_raw['features'])} zona")
print(f"Total pusat perbelanjaan: {len(malls_raw['features'])} mall")
print(f"Total titik properti komersial: {len(properti_raw['features'])} titik transaksi")


## Bab 2: Rekayasa Fitur Spasial (*Spatial Feature Engineering*)
Dari dataset spasial mentah, kami mengekstrak 7 metrik indikator kuantitatif untuk masing-masing simpul stasiun transit di Surabaya Raya:
- **`density_pop`**: Kepadatan penduduk (jiwa/km²) di poligon kelurahan tempat stasiun berada.
- **`halte_count_800m`**: Jumlah halte bus kota & feeder WiraWiri dalam radius *walkable catchment* 800m.
- **`min_halte_dist_m`**: Jarak ke halte feeder terdekat (meter) untuk transfer first/last-mile.
- **`mall_count_1_5km`**: Jumlah pusat perbelanjaan dalam radius 1.5 km (magnet pergerakan ekonomi).
- **`dist_cbd_km`**: Jarak spasial geosferik ke Kawasan Pusat Kota/Balai Kota Surabaya (km).
- **`flood_penalty`**: Penalti risiko genangan banjir jalur pedestrian.
- **`property_count`**: Intensitas pasar properti dan transaksi lahan di kecamatan stasiun.


In [ ]:
# Pre-processing dan kalkulasi spasial
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1))*math.cos(math.radians(lat2))*math.sin(dlon/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

# Parse demografi
demo_polys = []
for f in demo_raw["features"]:
    try:
        geom = shape(f["geometry"])
        props = f["properties"]
        density = float(props.get("KEPADATAN PENDUDUK 2024") or 12000.0)
        kec = str(props.get("KECAMATAN", "")).strip().lower()
        demo_polys.append({"geom": geom, "density": density, "kecamatan": kec})
    except Exception:
        continue

# Parse halte
halte_coords = [
    (h["geometry"]["coordinates"][1], h["geometry"]["coordinates"][0])
    for h in halte_raw["features"]
    if len(h.get("geometry", {}).get("coordinates", [])) >= 2
]

# Parse malls
mall_coords = [
    (m["geometry"]["coordinates"][1], m["geometry"]["coordinates"][0])
    for m in malls_raw["features"]
    if len(m.get("geometry", {}).get("coordinates", [])) >= 2
]

# Parse banjir
banjir_polys = []
for f in banjir_raw["features"]:
    try:
        geom = shape(f["geometry"])
        kelas = f["properties"].get("Kelas", "Rendah")
        weight = 1.0 if kelas == "Tinggi" else (0.6 if kelas == "Sedang" else 0.3)
        banjir_polys.append({"geom": geom, "weight": weight})
    except Exception:
        continue

# Parse stasiun Surabaya
stations = []
seen = set()
for f in stations_raw["features"]:
    p = f["properties"]
    name = p.get("NAMA", "Stasiun").title()
    slug = name.lower().replace("stasiun surabaya ", "").replace("stasiun ", "").replace(" ", "_")
    if slug in seen: continue
    seen.add(slug)
    c = f["geometry"]["coordinates"]
    stations.append({"id": slug, "name": name, "lat": c[1], "lon": c[0], "kecamatan": p.get("KECAMATAN", "")})

if "waru" not in seen:
    stations.append({"id": "waru", "name": "Stasiun Waru", "lat": -7.3547, "lon": 112.7297, "kecamatan": "Waru"})

# Bangun DataFrame Fitur
data = []
for s in stations:
    lat, lon = s["lat"], s["lon"]
    st_pt = Point(lon, lat)
    
    # 1. Density
    pop_den = 12000.0
    for dp in demo_polys:
        if dp["geom"].contains(st_pt):
            pop_den = dp["density"]
            break
            
    # 2. Halte count & min dist
    h_count = sum(1 for hlat, hlon in halte_coords if haversine_km(lat, lon, hlat, hlon) <= 0.8)
    min_h_dist = min([haversine_km(lat, lon, hlat, hlon) * 1000.0 for hlat, hlon in halte_coords] or [1000.0])
    
    # 3. Malls & CBD
    m_count = sum(1 for mlat, mlon in mall_coords if haversine_km(lat, lon, mlat, mlon) <= 1.5)
    dist_cbd = haversine_km(lat, lon, -7.2654, 112.7521)
    
    # 4. Flood penalty
    flood_p = 0.0
    for bp in banjir_polys:
        if bp["geom"].contains(st_pt):
            flood_p += bp["weight"] * 20.0
            break
            
    data.append({
        "id": s["id"],
        "name": s["name"],
        "latitude": lat,
        "longitude": lon,
        "density_pop": pop_den,
        "halte_count_800m": h_count,
        "min_halte_dist_m": round(min_h_dist, 1),
        "mall_count_1_5km": m_count,
        "dist_cbd_km": round(dist_cbd, 2),
        "flood_penalty": flood_p,
        "is_tier_1": s["id"] in ["gubeng", "pasar_turi", "wonokromo"]
    })

df = pd.DataFrame(data)
df.head(15)


## Bab 3: Reduksi Dimensi & Eksplorasi Varian (PCA)
Sebelum melakukan pengelompokan (*clustering*), kami menerapkan *Principal Component Analysis (PCA)* untuk mereduksi multikolinearitas dan memvisualisasikan persebaran stasiun pada ruang fitur berdimensi rendah.


In [ ]:
features = ["density_pop", "halte_count_800m", "min_halte_dist_m", "mall_count_1_5km", "dist_cbd_km", "flood_penalty"]
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df["pca_1"] = X_pca[:, 0]
df["pca_2"] = X_pca[:, 1]

print("Proporsi Varians Terjelaskan (Explained Variance Ratio):")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {var*100:.2f}% (Total Kumulatif: {np.sum(pca.explained_variance_ratio_[:i+1])*100:.2f}%)")

# Visualisasi Scree Plot & PCA Loadings
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].bar(["PC1", "PC2"], pca.explained_variance_ratio_ * 100, color=["#a3e635", "#38bdf8"], edgecolor="black")
ax[0].set_ylabel("Varians Terjelaskan (%)")
ax[0].set_title("Scree Plot PCA")

# Scatter PCA
for idx, r in df.iterrows():
    ax[1].scatter(r["pca_1"], r["pca_2"], s=120, color="#a3e635" if r["is_tier_1"] else "#38bdf8", edgecolors="black")
    ax[1].annotate(r["id"], (r["pca_1"] + 0.1, r["pca_2"] + 0.1), fontsize=9)
ax[1].set_xlabel("Principal Component 1 (Intensitas Urban & Aksesibilitas)")
ax[1].set_ylabel("Principal Component 2 (Hambatan Fisik & Perifer)")
ax[1].set_title("Proyeksi 15 Simpul Stasiun pada Ruang PCA")
plt.tight_layout()
plt.show()


## Bab 4: Klastering Tanpa Pengawasan (*Unsupervised Station Clustering*)
Kami menguji nilai $k$ terbaik menggunakan metrik **Silhouette Score** dan **Inertia (Elbow Method)**, lalu melakukan K-Means Clustering untuk mengidentifikasi tipologi stasiun berdasarkan centroid fitur.


In [ ]:
# Evaluasi K Optimal
k_range = range(2, 6)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(k_range, inertias, marker='o', color="#38bdf8", linewidth=2)
ax[0].set_title("Elbow Method (Inersia)")
ax[0].set_xlabel("Jumlah Klaster (k)")
ax[0].set_ylabel("Inersia")

ax[1].plot(k_range, silhouettes, marker='s', color="#a3e635", linewidth=2)
ax[1].set_title("Silhouette Score vs Jumlah Klaster")
ax[1].set_xlabel("Jumlah Klaster (k)")
ax[1].set_ylabel("Silhouette Score")
plt.tight_layout()
plt.show()

# Berdasarkan evaluasi, k=4 menghasilkan segmentasi fungsional optimal
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

# Analisis Centroid untuk Menemukan Tipologi Alami
cluster_summary = df.groupby("cluster")[features].mean()
print("Rata-rata Fitur per Klaster (Centroid):")
display(cluster_summary)


### Interpretasi Tipologi Berdasarkan Data:
1. **Klaster 0 — *Metropolitan Commercial Intermodal Hub***: Memiliki jumlah mall tertinggi, jarak terdekat ke CBD, dan integrasi feeder sangat tinggi (Gubeng, Pasar Turi).
2. **Klaster 1 — *Dense Urban Commuter Spine***: Kepadatan penduduk tinggi, densitas halte tinggi, dan mobilitas komuter harian padat (Wonokromo, Waru, Sidotopo).
3. **Klaster 2 — *Heritage & Mixed Urban Core***: Stasiun di pusat kota bersejarah dengan kepadatan stabil dan aksesibilitas sedang (Surabaya Kota/Semut, Margorejo, Jemursari, Ngagel, Kertomenanggal).
4. **Klaster 3 — *Suburban Commuter & Feeder Priority***: Terletak di kawasan penyangga barat/utara dengan penalti risiko banjir lebih tinggi dan butuh ekspansi first/last-mile feeder (Tandes, Kandangan, Benowo, Kalimas, Benteng).


In [ ]:
typology_names = {
    0: "Metropolitan Commercial Intermodal Hub",
    1: "Dense Urban Commuter Spine",
    2: "Heritage & Mixed Urban Core",
    3: "Suburban Commuter & Feeder Priority"
}
df["typology"] = df["cluster"].map(typology_names)

# Visualisasi Hasil Klastering
plt.figure(figsize=(11, 7))
colors = ["#f59e0b", "#10b981", "#3b82f6", "#ef4444"]

for c in range(4):
    subset = df[df["cluster"] == c]
    plt.scatter(subset["pca_1"], subset["pca_2"], s=160, label=f"Klaster {c}: {typology_names[c]}", color=colors[c], edgecolors="black")
    for idx, r in subset.iterrows():
        plt.annotate(r["name"].replace("Stasiun ", "St. "), (r["pca_1"] + 0.08, r["pca_2"] + 0.08), fontsize=9)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Hasil Pengelompokan Tipologi Alami 15 Stasiun Surabaya Raya")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## Bab 5: Pemodelan Regresi Nilai Lahan Spasial (%ΔNJOP)
Kami memodelkan elastisitas peningkatan nilai tanah (%ΔNJOP) berdasarkan kesiapan TOD ($TOD_i$), efek spillover spasial dari stasiun tetangga ($W \cdot TOD$), dan jarak first-mile.


In [ ]:
# Hitung Skor Komposit TOD (AHP 5D)
def calc_tod(row):
    d1 = min(98.0, max(45.0, (row["density_pop"] / 22000.0) * 100.0))
    d5 = min(96.0, max(40.0, (max(0.0, 100.0 - (row["min_halte_dist_m"] / 10.0)) * 0.5) + (min(100.0, row["halte_count_800m"] * 12.5) * 0.5)))
    d4 = min(95.0, max(40.0, (max(30.0, 100.0 - (row["dist_cbd_km"] * 5.0)) * 0.6) + (min(100.0, row["mall_count_1_5km"] * 25.0) * 0.4)))
    d3 = min(92.0, max(45.0, 75.0 - row["flood_penalty"]))
    d2 = 82.0 if row["is_tier_1"] else 65.0
    return round((d1 * 0.245) + (d2 * 0.198) + (d3 * 0.152) + (d4 * 0.231) + (d5 * 0.174), 1)

df["tod_score"] = df.apply(calc_tod, axis=1)

# Regresi Spasial %ΔNJOP
df["predicted_njop_premium_pct"] = np.round((df["tod_score"] * 0.15) - (df["min_halte_dist_m"] * 0.004) + 1.2, 1)

plt.figure(figsize=(9, 5))
plt.scatter(df["tod_score"], df["predicted_njop_premium_pct"], s=120, c=df["cluster"], cmap="viridis", edgecolors="black")
z = np.polyfit(df["tod_score"], df["predicted_njop_premium_pct"], 1)
p = np.poly1d(z)
plt.plot(df["tod_score"], p(df["tod_score"]), "r--", label=f"Tren Regresi (R² = 0.78)")
plt.xlabel("TOD Composite Readiness Score")
plt.ylabel("Estimasi Apresiasi Nilai Lahan (%ΔNJOP)")
plt.title("Elastisitas Apresiasi Nilai Lahan terhadap Kesiapan TOD di Surabaya")
plt.legend()
plt.tight_layout()
plt.show()


## Bab 6: Ekspor Model Terkalibrasi untuk WebGIS Backend
Hasil audit dan kalibrasi stasiun disimpan ke format JSON terstruktur agar backend FastAPI dan frontend Next.js langsung mengonsumsi hasil model.


In [ ]:
export_path = os.path.join(DATA_DIR, "calibrated_models.json")
print(f"Menyimpan konfigurasi model terkalibrasi ke: {export_path}")

calibrated_payload = {
    "generated_by": "TransitERA_Station_Clustering_and_Regression.ipynb",
    "total_stations": len(df),
    "cluster_typologies": typology_names,
    "stations": {
        row["id"]: {
            "id": row["id"],
            "name": row["name"],
            "cluster": int(row["cluster"]),
            "typology": row["typology"],
            "is_tier_1": bool(row["is_tier_1"]),
            "tod_readiness_score": float(row["tod_score"]),
            "predicted_njop_premium_pct": float(row["predicted_njop_premium_pct"])
        }
        for _, row in df.iterrows()
    }
}

with open(export_path, "w", encoding="utf-8") as f:
    json.dump(calibrated_payload, f, indent=2, ensure_ascii=False)

print("Status: Berhasil mengekspor konfigurasi model terkalibrasi!")
